# 03 — Model Training

Train Logistic Regression, Random Forest, XGBoost, and SVM. 5-fold CV, then evaluate on test set. Save metrics and figures (ROC curves, confusion matrix, model comparison).

In [ ]:
import sys
import os
ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.insert(0, os.path.join(ROOT, 'src'))
os.chdir(ROOT)

from preprocess import (
    load_raw_data,
    find_classification_column,
    create_target_and_filter,
    prepare_features_and_target,
    train_test_split_and_scale,
    FEATURE_COLS,
)
from train import train_all_models
from evaluate import (
    evaluate_all_models,
    plot_roc_curves,
    plot_confusion_matrix,
    plot_model_comparison,
    plot_cv_accuracy_box,
)

In [ ]:
# Preprocessing (same as notebook 02)
raw = load_raw_data(data_dir=os.path.join(ROOT, 'data', 'raw'))
class_col = find_classification_column(raw)
df = create_target_and_filter(raw, class_col)
feature_cols = [c for c in FEATURE_COLS if c in df.columns]
X, y = prepare_features_and_target(df, feature_cols=feature_cols, drop_pct_missing=0.30)
X_train, X_test, y_train, y_test, scaler = train_test_split_and_scale(
    X, y, test_size=0.2, random_state=42, use_smote=True
)

In [ ]:
# Train all 4 models with 5-fold CV
models, cv_results = train_all_models(X_train, y_train, cv=5)
metrics_df = evaluate_all_models(models, X_test, y_test, cv_results)
metrics_df.to_csv(os.path.join(ROOT, 'results', 'metrics_table.csv'), index=False)
metrics_df

In [ ]:
# Figures: ROC curves, model comparison, confusion matrix (best model), CV box
fig_dir = os.path.join(ROOT, 'figures')
plot_roc_curves(models, X_test, y_test.values, os.path.join(fig_dir, 'fig4_roc_curves.png'))
plot_model_comparison(metrics_df, os.path.join(fig_dir, 'fig3_model_comparison.png'))

best_name = metrics_df.loc[metrics_df['ROC_AUC'].idxmax(), 'Model']
best_model = models[best_name]
y_pred = best_model.predict(X_test)
plot_confusion_matrix(y_test.values, y_pred, os.path.join(fig_dir, 'fig5_confusion_matrix.png'))
plot_cv_accuracy_box(models, X_train, y_train, cv=5, save_path=os.path.join(fig_dir, 'fig6_cv_accuracy.png'))